# Notebook 08 — Transformer Training
Trains a Transformer encoder classifier on CIC-DDoS2019 with hyperparameter tuning.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import copy
import numpy as np
import torch
import yaml
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
from google.colab import drive
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/tugas-akhir-ai")
    SPLITS_DIR = DRIVE_ROOT / "splits"
else:
    import yaml
    with open("../config.yaml") as f:
        _cfg = yaml.safe_load(f)
    SPLITS_DIR = Path("..") / _cfg["data"]["splits_path"]
    DRIVE_ROOT = SPLITS_DIR.parent
print(f"SPLITS_DIR: {SPLITS_DIR}")

In [ ]:
with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

# Transformer uses same seq splits as LSTM/GRU
X_train = np.load(SPLITS_DIR / "X_train_seq.npy")
y_train = np.load(SPLITS_DIR / "y_train_seq.npy")
X_val   = np.load(SPLITS_DIR / "X_val_seq.npy")
y_val   = np.load(SPLITS_DIR / "y_val_seq.npy")
X_test  = np.load(SPLITS_DIR / "X_test_seq.npy")
y_test  = np.load(SPLITS_DIR / "y_test_seq.npy")

n_features = X_train.shape[2]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"n_features={n_features}")

## Baseline run (config defaults)

In [ ]:
from src.models.transformer_model import build_transformer
from src.train import train_model
from src.utils import log_experiment

model = build_transformer(cfg, n_features)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
history_baseline, ckpt_baseline = train_model(
    model, X_train, y_train, X_val, y_val,
    cfg=cfg, model_key='transformer',
    checkpoint_dir='../results/checkpoints',
)

log_experiment({
    'exp_id': 'transformer_01_baseline',
    'model': 'Transformer',
    'd_model': cfg['transformer']['d_model'],
    'nhead': cfg['transformer']['nhead'],
    'num_layers': cfg['transformer']['num_layers'],
    'dropout': cfg['transformer']['dropout'],
    'best_val_f1': round(max(history_baseline['val_f1']), 4),
    'notes': 'baseline — config defaults',
}, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

## Hyperparameter Experiments

In [ ]:
def run_transformer_experiment(exp_id, overrides, notes=''):
    exp_cfg = copy.deepcopy(cfg)
    exp_cfg['transformer'].update(overrides)

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    m = build_transformer(exp_cfg, n_features)
    hist, ckpt = train_model(
        m, X_train, y_train, X_val, y_val,
        cfg=exp_cfg, model_key='transformer',
        checkpoint_dir='../results/checkpoints',
    )
    best_f1 = round(max(hist['val_f1']), 4)
    log_experiment({
        'exp_id': exp_id,
        'model': 'Transformer',
        'd_model': exp_cfg['transformer']['d_model'],
        'nhead': exp_cfg['transformer']['nhead'],
        'num_layers': exp_cfg['transformer']['num_layers'],
        'dropout': exp_cfg['transformer']['dropout'],
        'best_val_f1': best_f1,
        'notes': notes,
    }, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))
    print(f'[{exp_id}] best_val_f1={best_f1}')
    return hist, ckpt, best_f1

In [ ]:
# Exp 02 — larger d_model (must be divisible by nhead)
hist_02, _, f1_02 = run_transformer_experiment('transformer_02_dmodel128', {'d_model': 128, 'dim_feedforward': 256}, 'd_model=128')

In [ ]:
# Exp 03 — more encoder layers
hist_03, _, f1_03 = run_transformer_experiment('transformer_03_layers4', {'num_layers': 4}, 'num_layers=4')

In [ ]:
# Exp 04 — more attention heads
hist_04, _, f1_04 = run_transformer_experiment('transformer_04_nhead8', {'d_model': 64, 'nhead': 8}, 'nhead=8')

In [ ]:
# Exp 05 — higher dropout
hist_05, _, f1_05 = run_transformer_experiment('transformer_05_dropout03', {'dropout': 0.3}, 'dropout=0.3')

In [ ]:
# Exp 06 — lower learning rate
hist_06, _, f1_06 = run_transformer_experiment('transformer_06_lr0001', {'learning_rate': 0.0001}, 'lr=0.0001')

## Final Evaluation on Test Set

In [ ]:
import pandas as pd
from src.evaluate import evaluate_dl_model
from src.utils import plot_loss_curves

results = pd.read_csv(str(DRIVE_ROOT / "metrics_summary.csv"))
tr_results = results[results['model'] == 'Transformer'].copy()
avail = [c for c in ['exp_id', 'd_model', 'nhead', 'num_layers', 'dropout', 'best_val_f1', 'notes'] if c in tr_results.columns]
print(tr_results[avail])

_hist_map = {
    'transformer_01_baseline':  history_baseline,
    'transformer_02_dmodel128': hist_02,
    'transformer_03_layers4':   hist_03,
    'transformer_04_nhead8':    hist_04,
    'transformer_05_dropout03': hist_05,
    'transformer_06_lr0001':    hist_06,
}
best_row = tr_results.loc[tr_results['best_val_f1'].idxmax()]
print(f'\nBest experiment: {best_row["exp_id"]}  val_F1={best_row["best_val_f1"]}')
best_history = _hist_map.get(best_row['exp_id'], history_baseline)

In [ ]:
best_model = build_transformer(cfg, n_features).to(device)
best_model.load_state_dict(torch.load('../results/checkpoints/best_transformer.pt', map_location=device))

y_pred_tr, y_prob_tr = evaluate_dl_model(
    best_model, X_test, y_test,
    model_name='Transformer',
    history=best_history,
    figures_dir='../results/figures',
    csv_path=str(DRIVE_ROOT / "metrics_summary.csv"),
)

np.save('../results/transformer_y_prob.npy', y_prob_tr)

In [ ]:
plot_loss_curves(
    best_history['train_loss'], best_history['val_loss'],
    title='Transformer Loss Curves (best)',
    save_path='../results/figures/loss_transformer.png',
)

In [ ]:
import shutil
shutil.copy('../results/checkpoints/best_transformer.pt', str(DRIVE_ROOT / 'best_transformer.pt'))
np.save(str(DRIVE_ROOT / 'transformer_y_prob.npy'), y_prob_tr)
print(f'Checkpoint + probs saved to Drive: {DRIVE_ROOT}')